# lenia-vlm on Colab

Runs the Lenia simulator on a GPU (PyTorch) with results saved to your Google Drive.

**Before the first run**
1. *Runtime → Change runtime type → GPU* (a T4 is fine).
2. *Secrets* (the key icon in the left sidebar): add `GITHUB_TOKEN`, a fine-grained GitHub token with
   read-only *Contents* access to `feli10/lenia-vlm`, and turn on notebook access. Only needed while
   the repo is private.
3. Optional, only for VLM judging: add `OPENROUTER_API_KEY` and `TYPESAFE_API_KEY` the same way.
   They are written to a `.env` on this temporary machine and never printed.

Run the cells top to bottom. Re-running the setup cells is harmless (they pull the latest code).

## 1. GPU and Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import os
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/lenia-runs'      # every result goes here and survives disconnects
os.makedirs(RUNS, exist_ok=True)
print('results folder:', RUNS)

## 2. Code, dependencies, data

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

REPO, DIR = 'github.com/feli10/lenia-vlm.git', '/content/lenia-vlm'
tok = secret('GITHUB_TOKEN')
url = f'https://x-access-token:{tok}@{REPO}' if tok else f'https://{REPO}'
if not os.path.isdir(DIR):
    r = subprocess.run(['git', 'clone', '-q', url, DIR], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', DIR, 'pull', '-q', url, 'main'], capture_output=True, text=True)
print('git:', 'ok' if r.returncode == 0 else r.stderr.replace(tok or '\0', '***'))
subprocess.run(['git', '-C', DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'])   # don't keep the token in .git
os.chdir(DIR)
!git log --oneline -1
!pip install -q -e .

In [ ]:
# Chan's catalogue (MIT licence), and API keys for the judges if you added them as secrets
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/animals.json'):
    !curl -sL -o data/animals.json https://raw.githubusercontent.com/Chakazul/Lenia/master/Python/animals.json
print('catalogue:', os.path.getsize('data/animals.json'), 'bytes')
keys = {k: secret(k) for k in ('OPENROUTER_API_KEY', 'TYPESAFE_API_KEY')}
with open('.env', 'w') as f:
    for k, v in keys.items():
        if v:
            f.write(f'{k}={v}\n')
print('judge keys present:', [k for k, v in keys.items() if v] or 'none (fine for simulation-only runs)')

## 3. Check the GPU simulator, then time it

Compares the PyTorch step with the numpy reference from identical starts, then times both. Do not
use the GPU for runs unless this ends with **ALL CHECKS PASSED**.

In [ ]:
!python -m lenia --backend auto bench

Optional: the full test suite (about a minute).

In [ ]:
!pip install -q pytest && python -m pytest -q tests

## 4. A first GPU run

A small multikernel survey at double resolution, straight to Drive. Compare its time with the same
command on your Mac to see the real end-to-end speedup (the bench above times the update step only;
the alive classifier still runs on the CPU).

In [ ]:
import time
t0 = time.time()
!python -m lenia --backend auto --runs {RUNS} --config configs/r26.yaml survey --mode multikernel --n 64 --out {RUNS}/gpu-test-survey
print(f'{time.time() - t0:.0f} s')

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{RUNS}/gpu-test-survey/sheet_alive.png'))[:1]:
    display(Image(p, width=700))
!cat {RUNS}/gpu-test-survey/summary.json | head -20

## 5. Experiment A: multi-channel creatures (run 2, alive v2)

Starts from Chan's multi-channel `found/` collections (files 221, 223, 231, 231b, 232, 233, 233c, 233s;
about 2,450 creatures), all brought to kernel radius 20 in a 256x256 world (`configs/found.yaml`).

1. **Download** the collections (small; re-run after a disconnect, the machine is wiped).
2. **Survey**: run every creature from its own pattern, keep the living ones, then calibrate the checks
   on them: each living creature is also run at 2x resolution, and its texture is measured
   (`calibration.json`, `sheet_texture.png`). All creatures, roughly 20-40 minutes on a T4.
3. **Search**: novelty search from 64 diverse living creatures that pass the calibrated checks, with
   the same hard checks for every child (alive at 100, still alive at 200, texture, coherent at 2x).
   It saves a checkpoint to Drive every 2 minutes: after a disconnect, re-run sections 1-2 and the
   same cell, and it continues. About 35-50 minutes.

Run 1 (each creature at Chan's own R, checks calibrated on the classic catalogue) is in
`lenia-runs/found-survey` and `lenia-runs/expA-explore`; this run writes new folders.


In [ ]:
!python -m lenia found download

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml found survey --out {RUNS}/found-survey-v2

In [ ]:
from IPython.display import Image, display
import glob, json
s = json.load(open(f'{RUNS}/found-survey-v2/summary.json'))
print(json.dumps({n: f"{f['alive']}/{f['creatures']}" for n, f in s['per_file'].items()}, indent=1))
print(json.dumps(s['calibration'], indent=1))
display(Image(f'{RUNS}/found-survey-v2/sheet_texture.png', width=700))
for p in sorted(glob.glob(f'{RUNS}/found-survey-v2/sheet_23*.png'))[:2]:
    display(Image(p, width=700))

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml explore --from-found {RUNS}/found-survey-v2 --roots 64 --evals 3200 --out {RUNS}/expA2-explore

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml explore --replot {RUNS}/expA2-explore --no-catalogue
for name in ('sheet_novel.png', 'sheet_lineages.png'):
    display(Image(f'{RUNS}/expA2-explore/{name}', width=700))

## 6. Phase 2: the labelling pool

Picks about 300 of Chan's living creatures (those passing the calibrated checks) and up to 10
descendants per lineage from the run-2 search, re-simulates each, renders the fixed-camera image the
judges see, holds out 20% of the creatures for testing, and makes the pairs. A few minutes on a T4.

The labelling itself (Luna, Jev) runs on your Mac, where the API keys are: see `docs/lenia.md`,
"Phase 2: labelling pool". Your Mac reads the pool straight from Drive.


In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml pool build

In [ ]:
from IPython.display import Image, display
import json
print(json.dumps(json.load(open(f'{RUNS}/pool/build.json')), indent=1)[:800])
for name in ('sheet_chan.png', 'sheet_search.png'):
    display(Image(f'{RUNS}/pool/{name}', width=700))

## Notes

- Every command takes `--backend auto --runs {RUNS}`; results then live in your Drive folder.
- The free tier disconnects idle sessions and caps sessions at about 12 hours. `lenia explore` and
  `lenia guided run` resume after a disconnect (re-run sections 1-2, then the same command with the
  same `--out`). After a disconnect the machine is wiped, your Drive is not.
- Claude can read your Drive folder: just say when a run has finished.